In [ ]:
# =====================================================================
# Ergodicity Validation v3 — KDE Weights + Multi-Height Grid
#   Fix 1: KDE-smoothed weights (not raw histogram)
#   Fix 2: 5 z-levels [1.5..2.0] matching RWP height U[1.5,2.0]
# =====================================================================

import torch, numpy as np, math, time
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# 1. System Parameters
# ============================================================
rx,ry,rz=10.0,10.0,3.0; xBs,yBs,zBs=10.0,-100.0,-10.0
Ea=8; dB=0.075; lam=0.075; L1=2; dref=abs(yBs)*1.5
Pd=40.0; Rth=0.2; Nd=-174.0; Bw=20e6; pm_val=0.2; nu=5
PB=10**(Pd/10)*1e-3; N0=10**(Nd/10)*1e-3*Bw

# ============================================================
# 2. RWP Trajectory Generator
# ============================================================
def generate_human_trajectory(sim_time,dt=10,n_users=5,room_x=10.0,room_y=10.0,rng=None):
    if rng is None: rng=np.random
    room_size=[0.0,room_x,0.0,room_y]
    hotspot_center=np.array([room_x/2,room_y/2]); hotspot_radius=room_x/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    time_steps=int(sim_time/dt)
    def gen_target(pos):
        if rng.rand()<g_h:
            t=hotspot_center+(rng.rand(2)-0.5)*2*hotspot_radius
        else:
            t=np.array([room_size[0]+rng.rand()*(room_size[1]-room_size[0]),
                        room_size[2]+rng.rand()*(room_size[3]-room_size[2])])
        t[0]=np.clip(t[0],room_size[0],room_size[1])
        t[1]=np.clip(t[1],room_size[2],room_size[3]);return t
    def tau(rgn): return tau_h if rgn=='hot' else tau_n
    def spd(rgn): return v_h if rgn=='hot' else v_n
    users_height=1.5+0.5*rng.rand(n_users)
    pos=np.zeros((n_users,time_steps,2))
    up=np.zeros((n_users,2));ur=[None]*n_users;us=[None]*n_users
    ut=np.zeros((n_users,2));ud=np.zeros((n_users,2));usp=np.zeros(n_users);uprem=np.zeros(n_users)
    for i in range(n_users):
        up[i]=[room_size[0]+rng.rand()*(room_size[1]-room_size[0]),room_size[2]+rng.rand()*(room_size[3]-room_size[2])]
        d2h=np.linalg.norm(up[i]-hotspot_center);ur[i]='hot' if d2h<=hotspot_radius else 'normal'
        if rng.rand()<p_t:
            us[i]='transfer';ut[i]=gen_target(up[i]);dv=ut[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=spd(ur[i])
        else:
            us[i]='pause';uprem[i]=rng.exponential(tau(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,time_steps):
        for i in range(n_users):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:
                    us[i]='transfer';ut[i]=gen_target(up[i]);dv=ut[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=spd(ur[i])
            else:
                md=usp[i]*dt;pd=ut[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut[i].copy();d2h=np.linalg.norm(up[i]-hotspot_center);ur[i]='hot' if d2h<=hotspot_radius else 'normal'
                    if rng.rand()<p_s:
                        us[i]='pause';uprem[i]=rng.exponential(tau(ur[i]))
                    else:
                        ut[i]=gen_target(up[i]);dv=ut[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=spd(ur[i])
                else:
                    up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((n_users*time_steps,3));idx=0
    for u in range(n_users):
        for s in range(time_steps):
            pts[idx]=[pos[u,s,0],pos[u,s,1],users_height[u]];idx+=1
    return pts

print('Trajectory generator ready.')

# ============================================================
# 3. Multi-Height Grid + KDE Weights
# ============================================================
GR=200
Z_HEIGHTS=[1.5,1.625,1.75,1.875,2.0]  # RWP height: U[1.5,2.0]
N_Z=len(Z_HEIGHTS)

xv=torch.linspace(0,rx,GR,device=device); yv=torch.linspace(0,ry,GR,device=device)
Xg,Yg=torch.meshgrid(xv,yv,indexing='ij')
xy_flat=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)  # [40000, 2]

# Build full multi-height grid: [N_Z * 40000, 3]
gl_list=[]
for zh in Z_HEIGHTS:
    gl_list.append(torch.stack([Xg.flatten(),Yg.flatten(),
                                 torch.full_like(Xg.flatten(),zh)],dim=1))
gl=torch.cat(gl_list,dim=0)  # [200000, 3]
Ngrid=gl.shape[0]

# KDE weights from 100,000s RWP (xy only, same weight for all z)
print('Building KDE weights (RWP 100,000s)...')
np.random.seed(777)
emp_traj=generate_human_trajectory(sim_time=100000,dt=10)
emp_xy_kde=emp_traj[:,:2].T  # [2, T]
kde=gaussian_kde(emp_xy_kde,bw_method='scott')
grid_xy_np=xy_flat.cpu().numpy().T  # [2, 40000]
w_xy=kde(grid_xy_np).flatten()  # [40000]
w_xy=w_xy/w_xy.sum()
# Replicate for each z level, then normalize
w_full=np.tile(w_xy,N_Z)
gw=torch.tensor(w_full/w_full.sum(),dtype=torch.float32,device=device)
print(f'  KDE weights ready, hotspot/edge ratio = {gw.max().item()/gw.min().item():.1f}x')
print(f'  Multi-height grid: {N_Z} levels x {GR}x{GR} = {Ngrid} points')

# NLoS params for full grid
np.random.seed(42)
_ps=torch.tensor(2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_et=torch.tensor(10**((-15+5*np.random.rand(Ngrid))/10),dtype=torch.float32,device=device)

# ============================================================
# 4. Channel Model
# ============================================================
def phys_chan(wp,lc):
    if not isinstance(wp,torch.Tensor): wp=torch.tensor(wp,dtype=torch.float32,device=device)
    if wp.dim()==1: wp=wp.unsqueeze(0)
    Bn=wp.shape[0]; xc,zc,Lx,Lz=wp[:,0],wp[:,1],wp[:,2],wp[:,3]
    xu,yu,zu=lc[:,0],lc[:,1],lc[:,2]
    dx=xc-xBs; dy=torch.full_like(xc,0.0-yBs); dz=zc-zBs
    Rw=torch.sqrt(dx**2+dy**2+dz**2); th=torch.atan2(dy,dx); ph=torch.acos(dz/Rw)
    kx=torch.sin(ph)*torch.cos(th); kz=torch.cos(ph)
    x1=xc-Lx/2;z1=zc-Lz/2;x2=xc-Lx/2;z2=zc+Lz/2;x3=xc+Lx/2;z3=zc-Lz/2;x4=xc+Lx/2;z4=zc+Lz/2
    def rd(xs,zs):
        ddx=xs.unsqueeze(1)-xBs;ddy=torch.full_like(ddx,0.0-yBs);ddz=zs.unsqueeze(1)-zBs
        L=torch.sqrt(ddx**2+ddy**2+ddz**2);return ddx/L,ddy/L,ddz/L
    ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
    dux=xu-xBs;duy=yu-yBs;duz=zu-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2)
    uux=dux/Lu;uuz=duz/Lu
    ua=torch.stack([ux1,ux2,ux3,ux4],0);uz=torch.stack([uz1,uz2,uz3,uz4],0)
    umin=ua.min(0).values;umax=ua.max(0).values;zmin=uz.min(0).values;zmax=uz.max(0).values
    b=1000.0
    ix=torch.sigmoid(b*(uux-umin))*torch.sigmoid(b*(umax-uux))
    iz=torch.sigmoid(b*(uuz-zmin))*torch.sigmoid(b*(zmax-uuz))
    il=ix*iz
    d2x=xu-xc.unsqueeze(1);d2y=yu;d2z=zu-zc.unsqueeze(1)
    Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
    ax=(1-il)*(kx.unsqueeze(1)-t1);az=(1-il)*(kz.unsqueeze(1)-t2)
    sx=torch.sinc((math.pi/lam)*Lx.unsqueeze(1)*ax);sz=torch.sinc((math.pi/lam)*Lz.unsqueeze(1)*az)
    na=torch.arange(Ea,dtype=torch.float32,device=device)
    p1=(2*math.pi/lam)*dB*na*torch.sin(th).unsqueeze(1)
    a1=(1/math.sqrt(Ea))*torch.complex(torch.cos(p1),torch.sin(p1))
    v1m=(lam/(4*math.pi))/Rw;v1p=-(2*math.pi/lam)*Rw
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p));H1=v1.unsqueeze(1)*a1.conj()
    p2=(2*math.pi/lam)*dB*na.unsqueeze(0)*torch.sin(_tt).unsqueeze(1)
    a2=(1/math.sqrt(Ea))*torch.complex(torch.cos(p2),torch.sin(p2))
    v2m=_et*(lam/(4*math.pi*dref));v2=torch.complex(v2m*torch.cos(_ps),v2m*torch.sin(_ps))
    H2=v2.unsqueeze(1)*a2.conj().unsqueeze(0)
    Ht=math.sqrt(Ea/L1)*(H1.unsqueeze(1)+H2)
    fm=(Lx.unsqueeze(1)*Lz.unsqueeze(1)*sx*sz)/(lam*Ru)
    fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
    fc=torch.complex(fm*torch.cos(fp.unsqueeze(1)),fm*torch.sin(fp.unsqueeze(1)))
    return Ht*fc.unsqueeze(2)

def phys_chan_dynamic(wp,lc,ps,tt,et):
    if wp.dim()==1: wp=wp.unsqueeze(0)
    Bn=wp.shape[0]; xc,zc,Lx,Lz=wp[:,0],wp[:,1],wp[:,2],wp[:,3]
    xu,yu,zu=lc[:,0],lc[:,1],lc[:,2]
    dx=xc-xBs; dy=torch.full_like(xc,0.0-yBs); dz=zc-zBs
    Rw=torch.sqrt(dx**2+dy**2+dz**2); th=torch.atan2(dy,dx); ph=torch.acos(dz/Rw)
    kx=torch.sin(ph)*torch.cos(th); kz=torch.cos(ph)
    x1=xc-Lx/2;z1=zc-Lz/2;x2=xc-Lx/2;z2=zc+Lz/2;x3=xc+Lx/2;z3=zc-Lz/2;x4=xc+Lx/2;z4=zc+Lz/2
    def rd(xs,zs):
        ddx=xs.unsqueeze(1)-xBs;ddy=torch.full_like(ddx,0.0-yBs);ddz=zs.unsqueeze(1)-zBs
        L=torch.sqrt(ddx**2+ddy**2+ddz**2);return ddx/L,ddy/L,ddz/L
    ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
    dux=xu-xBs;duy=yu-yBs;duz=zu-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2)
    uux=dux/Lu;uuz=duz/Lu
    ua=torch.stack([ux1,ux2,ux3,ux4],0);uz=torch.stack([uz1,uz2,uz3,uz4],0)
    umin=ua.min(0).values;umax=ua.max(0).values;zmin=uz.min(0).values;zmax=uz.max(0).values
    b=1000.0
    ix=torch.sigmoid(b*(uux-umin))*torch.sigmoid(b*(umax-uux))
    iz=torch.sigmoid(b*(uuz-zmin))*torch.sigmoid(b*(zmax-uuz))
    il=ix*iz
    d2x=xu-xc.unsqueeze(1);d2y=yu;d2z=zu-zc.unsqueeze(1)
    Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
    ax=(1-il)*(kx.unsqueeze(1)-t1);az=(1-il)*(kz.unsqueeze(1)-t2)
    sx=torch.sinc((math.pi/lam)*Lx.unsqueeze(1)*ax);sz=torch.sinc((math.pi/lam)*Lz.unsqueeze(1)*az)
    na=torch.arange(Ea,dtype=torch.float32,device=device)
    p1=(2*math.pi/lam)*dB*na*torch.sin(th).unsqueeze(1)
    a1=(1/math.sqrt(Ea))*torch.complex(torch.cos(p1),torch.sin(p1))
    v1m=(lam/(4*math.pi))/Rw;v1p=-(2*math.pi/lam)*Rw
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p));H1=v1.unsqueeze(1)*a1.conj()
    p2=(2*math.pi/lam)*dB*na.unsqueeze(0)*torch.sin(tt).unsqueeze(1)
    a2=(1/math.sqrt(Ea))*torch.complex(torch.cos(p2),torch.sin(p2))
    v2m=et*(lam/(4*math.pi*dref));v2=torch.complex(v2m*torch.cos(ps),v2m*torch.sin(ps))
    H2=v2.unsqueeze(1)*a2.conj().unsqueeze(0)
    Ht=math.sqrt(Ea/L1)*(H1.unsqueeze(1)+H2)
    fm=(Lx.unsqueeze(1)*Lz.unsqueeze(1)*sx*sz)/(lam*Ru)
    fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
    fc=torch.complex(fm*torch.cos(fp.unsqueeze(1)),fm*torch.sin(fp.unsqueeze(1)))
    return Ht*fc.unsqueeze(2)

# ============================================================
# 5. Evaluation Functions
# ============================================================
@torch.no_grad()
def outage_grid(wp):
    if isinstance(wp,np.ndarray):wp=torch.tensor(wp,dtype=torch.float32,device=device)
    if wp.dim()==1:wp=wp.unsqueeze(0)
    He=phys_chan(wp,gl);Hw=torch.sum(He,dim=2)/math.sqrt(Ea)
    dp=(torch.abs(Hw)**2)*pm_val*PB;it=(nu-1)*dp;sr=dp/(it+N0)
    bits=(torch.log2(1+sr)<Rth).float()
    # Self-blockage (Ref C.3)
    pblk=1.0/6.0
    bits=(1-pblk)*bits+pblk
    return torch.sum(bits*gw,dim=1).cpu().numpy()

@torch.no_grad()
def outage_trajectory(wp,traj_pts):
    if isinstance(wp,np.ndarray):wp=torch.tensor(wp,dtype=torch.float32,device=device)
    if wp.dim()==1:wp=wp.unsqueeze(0)
    lc=torch.tensor(traj_pts,dtype=torch.float32,device=device)
    T=len(lc)
    nlos_ps=torch.tensor(2*np.pi*np.random.rand(T),dtype=torch.float32,device=device)
    nlos_tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(T),dtype=torch.float32,device=device)
    nlos_et=torch.tensor(10**((-15+5*np.random.rand(T))/10),dtype=torch.float32,device=device)
    He=phys_chan_dynamic(wp,lc,nlos_ps,nlos_tt,nlos_et)
    Hw=torch.sum(He,dim=2)/math.sqrt(Ea)
    dp=(torch.abs(Hw)**2)*pm_val*PB;it=(nu-1)*dp;sr=dp/(it+N0)
    bits=(torch.log2(1+sr)<Rth).float()
    # Self-blockage (Ref C.3)
    pblk=1.0/6.0
    bits=(1-pblk)*bits+pblk
    return bits.mean().item()

# ============================================================
# 6. Knee Solution
# ============================================================
knee=np.array([4.809,1.070,0.100,0.207])
knee_grid=outage_grid(knee)[0]
print(f'\nKnee [{knee[0]:.3f},{knee[1]:.3f},{knee[2]:.3f},{knee[3]:.3f}]')
print(f'  Grid integral (KDE + {N_Z} heights) = {knee_grid*100:.2f}%')

# ============================================================
# EXPERIMENT 1
# ============================================================
sim_times=[300,500,1000,2000,3000,5000,8000,12000,16000,20000]
n_seeds=10
print(f'\n{"="*60}')
print('Exp 1: Ergodicity Convergence (KDE + Multi-Height)')
print('='*60)
exp1_means=[];exp1_stds=[]
for st in sim_times:
    outs=[]
    for sd in range(n_seeds):
        np.random.seed(1000+sd)
        traj=generate_human_trajectory(sim_time=st,dt=10)
        outs.append(outage_trajectory(knee,traj))
    outs=np.array(outs);exp1_means.append(outs.mean());exp1_stds.append(outs.std())
    print(f'  sim_time={st:5d}s  |  mean={outs.mean()*100:.2f}%  std={outs.std()*100:.2f}%')
exp1_means=np.array(exp1_means);exp1_stds=np.array(exp1_stds)

fig1,ax1=plt.subplots(figsize=(10,5))
ax1.fill_between(sim_times,(exp1_means-exp1_stds)*100,(exp1_means+exp1_stds)*100,
                 alpha=0.2,color='steelblue',label='+-1 sigma')
ax1.plot(sim_times,exp1_means*100,'o-',color='steelblue',linewidth=1.5,markersize=6,label='Markov mean')
ax1.axhline(y=knee_grid*100,color='red',linestyle='--',linewidth=2,label=f'Grid = {knee_grid*100:.2f}%')
ax1.set_xlabel('RWP Time [s]');ax1.set_ylabel('Outage [%]')
ax1.set_title('Exp 1: Ergodicity Convergence (KDE + Multi-Height)');ax1.legend();ax1.grid(alpha=0.3)
ax1.set_xscale('log')
fd=abs(exp1_means[-1]-knee_grid)*100
ax1.annotate(f'Delta={fd:.2f}%',xy=(sim_times[-1],exp1_means[-1]*100),fontsize=10,fontweight='bold')
plt.tight_layout();plt.savefig('fig_exp1_convergence.png',dpi=150);plt.show()
print(f'\nConvergence: Markov={exp1_means[-1]*100:.2f}%  Grid={knee_grid*100:.2f}%  Delta={fd:.3f}%')

# ============================================================
# EXPERIMENT 2
# ============================================================
print(f'\n{"="*60}')
print('Exp 2: Landscape Discrepancy (100 configs)')
print('='*60)
np.random.seed(99);N_config=100
lb=np.array([0.2,0.2,0.1,0.1]);ub=np.array([9.8,2.8,9.8,2.8])
c_lhs=lb+np.random.rand(60,4)*(ub-lb)
c_par=np.column_stack([np.random.uniform(4.5,5.5,40),np.random.uniform(0.9,1.2,40),
                        np.random.uniform(0.08,3.0,40),np.random.uniform(0.1,0.8,40)])
configs=np.vstack([c_lhs,c_par])[:N_config]
grid_outs=outage_grid(configs)
MT=5000;MS=5
print(f'{N_config} configs, RWP({MT}s,{MS} seeds)...')
markov_outs=np.zeros(N_config)
for ci in range(N_config):
    oi=[]
    for sd in range(MS):
        np.random.seed(20000+ci*100+sd)
        traj=generate_human_trajectory(sim_time=MT,dt=10)
        oi.append(outage_trajectory(configs[ci],traj))
    markov_outs[ci]=np.mean(oi)
    if (ci+1)%20==0:print(f'  ...{ci+1}/{N_config}')

mae=np.abs(grid_outs-markov_outs).mean()
ss_res=((grid_outs-markov_outs)**2).sum();ss_tot=((markov_outs-markov_outs.mean())**2).sum()
r2=1-ss_res/ss_tot;diffs=(grid_outs-markov_outs)*100

fig2,axes2=plt.subplots(1,2,figsize=(14,5.5))
ax=axes2[0]
ax.scatter(markov_outs*100,grid_outs*100,c='steelblue',s=15,alpha=0.6,edgecolors='white',linewidth=0.5)
ax.plot([0,100],[0,100],'r--',linewidth=1.5);ax.set_xlabel('Markov RWP Outage [%]');ax.set_ylabel('Grid Integral Outage [%]')
ax.set_title(f'Exp 2: Grid vs Markov (KDE+Multi-Height)\nMAE={mae*100:.2f}%  R2={r2:.4f}');ax.legend();ax.grid(alpha=0.3)
ax=axes2[1]
ax.hist(diffs,bins=30,color='steelblue',alpha=0.7,edgecolor='white')
ax.axvline(x=0,color='red',linestyle='--',linewidth=1.5)
ax.axvline(x=diffs.mean(),color='green',linewidth=1.5,label=f'Bias={diffs.mean():.3f}%')
ax.set_xlabel('Grid - Markov [%]');ax.set_title(f'Error (std={diffs.std():.2f}%)');ax.legend();ax.grid(alpha=0.3)
plt.tight_layout();plt.savefig('fig_exp2_discrepancy.png',dpi=150);plt.show()

print('\n'+'='*70)
print('VALIDATION SUMMARY (KDE + Multi-Height)')
print('='*70)
print(f'Exp 1: Markov(20ks)->Grid Delta = {fd:.3f}%')
print(f'Exp 2: MAE={mae*100:.2f}%  R2={r2:.4f}  Bias={diffs.mean():.3f}%')
print('='*70)

from IPython.display import FileLink,display
display(FileLink('fig_exp1_convergence.png'));display(FileLink('fig_exp2_discrepancy.png'))